In [13]:
import numpy as np
import pandas as pd
import sys
import os

sys.path.append('..')

from rl.action_space import INDEX_TO_ACTION, Strategy, get_action_strategy, get_action_count
from rl.action_masking import get_action_mask, get_valid_actions
from rl.state_builder import STATE_COLUMNS

## 1. Action Space Overview

In [14]:
print(f"Total Actions: {get_action_count()}")
for i, name in INDEX_TO_ACTION.items():
    strategy = get_action_strategy(i)
    print(f"{i:2}: {name:<25} | Strategy: {strategy.name}")

Total Actions: 17
 0: route_bug                 | Strategy: ROUTE
 1: route_ml_module           | Strategy: ROUTE
 2: route_build_infra         | Strategy: ROUTE
 3: route_docs                | Strategy: ROUTE
 4: route_billing             | Strategy: ROUTE
 5: route_product             | Strategy: ROUTE
 6: ask_uncertainty           | Strategy: CLARIFY
 7: ask_error_type            | Strategy: CLARIFY
 8: ask_version               | Strategy: CLARIFY
 9: ask_platform              | Strategy: CLARIFY
10: ask_hardware              | Strategy: CLARIFY
11: ask_vague_request         | Strategy: CLARIFY
12: suggest_top1              | Strategy: SUGGEST
13: suggest_top2              | Strategy: SUGGEST
14: suggest_top3              | Strategy: SUGGEST
15: suggest_high_conf_only    | Strategy: SUGGEST
16: escalate_human            | Strategy: ESCALATE


## 2. Masking Logic Verification


In [15]:
def test_mask(description, state_mods):
    state = np.zeros(len(STATE_COLUMNS))
    # Set defaults to enable most actions
    state[6] = 1.0  # SLA remaining
    state[21] = 1.0 # completeness
    state[23] = 1.0 # needs clarify
    
    for idx, val in state_mods.items():
        state[idx] = val
        
    valid_actions = get_valid_actions(state)
    valid_names = [INDEX_TO_ACTION[i] for i in valid_actions]
    
    print(f"--- {description} ---")
    print(f"Valid Actions ({len(valid_actions)}): {valid_names[:5]}..." if len(valid_names) > 5 else f"Valid Actions: {valid_names}")
    return valid_actions

# Scenario A: Everything valid
test_mask("Baseline (All valid)", {})

# Scenario B: Knowledge Gap
test_mask("Knowledge Gap (Disable Suggest)", {34: 1.0})

# Scenario C: High Frustration
test_mask("High Frustration (Disable Clarify)", {27: 0.8})

# Scenario D: Low SLA
test_mask("Low SLA (Disable Clarify)", {6: 0.1})

None

--- Baseline (All valid) ---
Valid Actions (17): ['route_bug', 'route_ml_module', 'route_build_infra', 'route_docs', 'route_billing']...
--- Knowledge Gap (Disable Suggest) ---
Valid Actions (13): ['route_bug', 'route_ml_module', 'route_build_infra', 'route_docs', 'route_billing']...
--- High Frustration (Disable Clarify) ---
Valid Actions (11): ['route_bug', 'route_ml_module', 'route_build_infra', 'route_docs', 'route_billing']...
--- Low SLA (Disable Clarify) ---
Valid Actions (11): ['route_bug', 'route_ml_module', 'route_build_infra', 'route_docs', 'route_billing']...


## Percentage of action masked per state

In [16]:
import pandas as pd
import numpy as np

from rl.action_masking import get_action_mask
from rl.state_builder import STATE_COLUMNS

# Load dataset
df = pd.read_csv("../data/final/rl_ready_dataset.csv")

# Extract state vectors
states = df[STATE_COLUMNS].values

In [17]:

mask_ratio = np.mean([1 - get_action_mask(s).mean() for s in states])
print("Avg masked ratio:", mask_ratio)

Avg masked ratio: 0.37241086
